# Demo 11 — GLMM with binomial distribution

This demo uses the `bacteria` dataset (MASS), which tracks the presence or absence
of H. influenzae in 50 children with otitis media, measured at five time points
(weeks 0, 2, 4, 6, 11) under three treatments (placebo, drug, drug+) — 220
observations in all. It asks whether treatment and time affect the probability
that bacteria are still present.

The outcome is binary, so its mean is a probability bounded between 0 and 1 and a
gaussian model is invalid. A binomial GLMM with a logit link, plus a random
intercept per child for the repeated measures, models it correctly:

    present ~ trt + week + (1 | ID)

The log-odds are estimated on the logit scale and the estimated marginal means are
back-transformed to probabilities — exactly the kind of binary outcome that
classical t-tests and ANOVA cannot handle at all.

## Setup

In [ ]:
import os
try:
    notebook_dir = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    notebook_dir = os.getcwd()  # JupyterLab sets CWD to the notebook directory
os.chdir(notebook_dir)

import matplotlib.pyplot as plt
from kbstatpy import Kbstat, KbstatOptions

## Options

`distribution = 'binomial'` with `link = 'logit'` for binary outcome. Two fixed-effect factors (treatment and week) with a random intercept per child.

In [ ]:
options = KbstatOptions()
options.in_file      = os.path.join(notebook_dir, '../data/bacteria.csv')
options.out_dir  = ''   # empty: show results inline only; set a folder to also save them
options.y            = 'present'
options.x            = 'trt, week'
options.id           = 'ID'
options.distribution = 'binomial'
options.link         = 'logit'
options.rename       = 'present -> Bacteria_present; trt -> Treatment; week -> Week'

## Model fitting

`fit()` estimates the model parameters via restricted maximum likelihood (REML).

In [ ]:
kb = Kbstat(options)
kb.fit()
print(f'Formula : {kb._build_formula()}')
print(f'AIC     : {kb.AIC:.3f}')
print(f'BIC     : {kb.BIC:.3f}')
print(f'logLik  : {kb.logLik:.3f}')

## ANOVA table (Type III)

In [ ]:
kb.anova()
kb.anova_table

## Post-hoc pairwise comparisons

Pairwise comparisons for treatment. EMMs are on the probability scale (back-transformed from logit).

In [ ]:
kb.posthoc()
kb.posthoc_table

## Data plot

Violin + jitter scatter with EMM and 95 % CI overlaid.

> An interactive version with hover tooltips is written as an HTML file when `out_dir` is set.

In [ ]:
kb.plot_data()
plt.show()

## Diagnostic plots

Six panels checking model assumptions.

See [STATISTICAL_NOTES.md](../../STATISTICAL_NOTES.md) for interpretation guidance.

In [ ]:
kb.plot_diagnostics()
plt.show()

## Save results

With `out_dir` set, `kb.save()` writes all tables, the summary, and the figures there; it is empty here, so this only shows a notice.

In [ ]:
kb.save()

## Interpretation

- EMMs represent estimated probabilities of bacterial presence for each treatment group, averaged over weeks.
- A significant treatment effect indicates that at least one treatment differs from the others in bacterial clearance.
- For a binomial GLMM, Satterthwaite df are undefined — asymptotic (z-based) inference is used instead.